# The Lost-in-the-Middle Effect [Step 01.04]

> **MLCourse - Agentic AI - Agent Patterns**

There is a comfortable assumption behind every "just use a bigger context window"
decision:

> If the fact is in the window, the model will use it.

**It is not true.** Liu et al. (2023), *Lost in the Middle*, showed that model
accuracy depends strongly on *where* in the input the needed fact sits, forming a
U-shaped curve: strong at the beginning, strong at the end, weakest in the middle.

```
  accuracy
     ^
   1 |*                                       *
     | *                                     *
     |   *                                 *
     |      *                          *
     |          * * * * * * * * * *
   0 +-------------------------------------------> position of the fact
      start                 middle                 end
```

### What you'll learn

- How to run this experiment yourself, correctly.
- What the effect looks like on **this** model, measured today, with real numbers.
- The three practical consequences for how you order context.

### Why it matters

If retrieval places the answer at rank 8 of 15, you may have retrieved perfectly
and still get a wrong answer. Reranking (`03_rag_advanced/11_reranking`) is
usually sold as "better retrieval"; a large part of its real value is simply
**moving the good document to position 1**, where the model will actually read it.

### Prerequisites

- [03_trimming_strategies](03_trimming_strategies.ipynb)
- [03_rag_advanced/11_reranking](../../03_rag_advanced/11_reranking)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. Designing the experiment properly

This experiment is easy to run badly. Four rules:

1. **One needle, many distractors.** The answer must appear exactly once.
2. **The distractors must be plausible.** Same format, same topic, same length.
   Random noise is trivially ignored and gives a flat, meaningless curve.
3. **The needle must not be guessable.** Use invented facts, so the model cannot
   answer from pretraining. We use fictional employee records with random codes.
4. **Vary only the position.** Same needle, same distractors, same question -
   only the index changes.

Then repeat each position several times and average, because a single sample of
a stochastic system tells you nothing.

In [2]:
import random

random.seed(11)

# 15 fictional records. Identical shape, so position is the only variable.
NAMES = ["Iris Kovac", "Dario Menzel", "Sunila Rao", "Petr Havel", "Noor Aziz",
         "Tomas Ferreira", "Greta Lindqvist", "Amara Diallo", "Ravi Menon",
         "Elif Yilmaz", "Jonas Brandt", "Mei Sasaki", "Ola Nowak",
         "Hugo Almeida", "Zara Haddad"]
TEAMS = ["Platform", "Billing", "Search", "Mobile", "Data", "Growth", "Security",
         "Payments", "Infra", "Design", "Support", "Quality", "Research",
         "Docs", "Identity"]

def make_records(n=15):
    recs = []
    for i in range(n):
        recs.append("Employee record %02d: %s works on the %s team and their "
                    "internal access code is %s-%03d."
                    % (i + 1, NAMES[i], TEAMS[i],
                       "".join(random.choice("BCDFGHJKLMNPQRSTVWXZ") for _ in range(3)),
                       random.randint(100, 999)))
    return recs


DISTRACTORS = make_records(15)

# The needle: one extra record, with the code we will ask about.
NEEDLE_NAME = "Wendell Ott"
NEEDLE_CODE = "QRV-742"
NEEDLE = ("Employee record XX: %s works on the Logistics team and their "
          "internal access code is %s." % (NEEDLE_NAME, NEEDLE_CODE))

QUESTION = ("What is the internal access code for %s? "
            "Reply with ONLY the code, nothing else." % NEEDLE_NAME)

print(NEEDLE)
print()
print("distractor example:", DISTRACTORS[0])
print("total distractors :", len(DISTRACTORS))
print("approx tokens for the full document set:",
      approx_tokens("\n".join(DISTRACTORS + [NEEDLE])))

Employee record XX: Wendell Ott works on the Logistics team and their internal access code is QRV-742.

distractor example: Employee record 01: Iris Kovac works on the Platform team and their internal access code is SWS-562.
total distractors : 15
approx tokens for the full document set: 395


> **Note the scale.** This is a *small* context by 2020s standards - about 500
> tokens. If the effect shows up here, it is far more severe at 50,000 tokens.
> We keep it small because the Groq free tier gives us 8000 tokens per minute and
> this experiment makes many calls.

In [3]:
def build_context(position: int) -> str:
    """Insert the needle at `position` among the distractors."""
    docs = DISTRACTORS[:position] + [NEEDLE] + DISTRACTORS[position:]
    # Renumber every record so the numbering is consistent and the needle's
    # position is not given away by an odd label.
    return "\n".join("Employee record %02d:%s" % (i + 1, d.split(":", 1)[1])
                     for i, d in enumerate(docs))


print(build_context(0).split("\n")[0])
print("...")
print(build_context(15).split("\n")[-1])

Employee record 01: Wendell Ott works on the Logistics team and their internal access code is QRV-742.
...
Employee record 16: Wendell Ott works on the Logistics team and their internal access code is QRV-742.


### 2. Run it

Five positions (first, quarter, middle, three-quarter, last), three trials each,
at temperature 0. Fifteen calls, each roughly 550 input tokens - the `chat()`
helper paces them so we stay under the free-tier ceiling.

In [4]:
SYSTEM = ("You answer questions using ONLY the employee records provided. "
          "Reply with the exact access code and nothing else.")

POSITIONS = [0, 4, 8, 12, 16]        # 16 distractors -> 16 possible slots
TRIALS = 3

raw = []
t_start = time.time()
for pos in POSITIONS:
    ctx = build_context(pos)
    for trial in range(TRIALS):
        out = chat([("system", SYSTEM),
                    ("user", "Employee records:\n%s\n\n%s" % (ctx, QUESTION))],
                   temperature=0.0, max_tokens=24)
        answer = out.content.strip()
        ok = NEEDLE_CODE.lower() in answer.lower()
        raw.append({"position": pos, "trial": trial, "answer": answer,
                    "correct": ok,
                    "in_tokens": out.usage_metadata["input_tokens"]})
        print("pos %2d trial %d -> %-14s %s" % (pos, trial, answer[:14],
                                                "OK" if ok else "WRONG"))

print("\n%d calls in %.0f seconds" % (len(raw), time.time() - t_start))

pos  0 trial 0 -> QRV-742        OK


pos  0 trial 1 -> QRV-742        OK


pos  0 trial 2 -> QRV-742        OK


pos  4 trial 0 -> QRV-742        OK


pos  4 trial 1 -> QRV-742        OK


...

pos 12 trial 1 -> QRV-742        OK


pos 12 trial 2 -> QRV-742        OK


pos 16 trial 0 -> QRV-742        OK


pos 16 trial 1 -> QRV-742        OK


pos 16 trial 2 -> QRV-742        OK

15 calls in 124 seconds


### Results


In [ ]:
n_pos = len(POSITIONS)
print("%-10s %-12s %10s   %s" % ("position", "where", "accuracy", "trials"))
print("-" * 58)
acc = {}
for i, pos in enumerate(POSITIONS):
    hits = [r["correct"] for r in raw if r["position"] == pos]
    a = sum(hits) / len(hits)
    acc[pos] = a
    where = ["first", "early", "MIDDLE", "late", "last"][i]
    bar = "#" * int(a * 20)
    print("%-10d %-12s %9.0f%%   %-22s %s"
          % (pos, where, a * 100, "".join("O" if h else "." for h in hits), bar))

print("-" * 58)
print("overall accuracy: %.0f%%" % (100 * sum(r["correct"] for r in raw) / len(raw)))
print("mean input tokens per call: %d" % (sum(r["in_tokens"] for r in raw) / len(raw)))


### 3. Reading the result honestly

Two possible outcomes, and **both are worth understanding**:

**If the middle positions scored lower** - you have reproduced the effect at a
tiny scale. Take it as a floor, not a ceiling: the published curves are measured
at 10-30x this context length and are much steeper there.

**If accuracy is flat at or near 100%** - that is the *expected* result for a
strong recent model on a 500-token context, and it is a genuinely useful
negative finding. The effect is a function of how full the window is, not a
constant property of the model. A 500-token haystack is simply not a haystack.

Either way, the honest conclusion is the same and it is not "the effect is fake":

> Position is a variable that affects accuracy, and it costs you nothing to put
> the important material where the model reads it best.

Below we print exactly what we measured, with no interpretation added.

In [6]:
first_last = (acc[POSITIONS[0]] + acc[POSITIONS[-1]]) / 2
middle = acc[POSITIONS[len(POSITIONS) // 2]]

print("MEASURED, this run, %s, %d distractors, %d trials/position:"
      % (GROQ_MODEL, len(DISTRACTORS), TRIALS))
print()
print("  accuracy at the edges (first & last) : %.0f%%" % (first_last * 100))
print("  accuracy in the middle               : %.0f%%" % (middle * 100))
print("  gap                                  : %+.0f points" % ((middle - first_last) * 100))
print()
if middle < first_last:
    print("  -> the middle was WORSE here: the effect reproduced at this scale.")
elif middle == first_last == 1.0:
    print("  -> flat at ceiling. This context (%d tokens) is too small to stress the"
          % approx_tokens(build_context(0)))
    print("     model. Reported honestly: no effect observed at this size.")
else:
    print("  -> no clear positional pattern at this scale.")

MEASURED, this run, qwen/qwen3.8-27b, 15 distractors, 3 trials/position:

  accuracy at the edges (first & last) : 100%
  accuracy in the middle               : 100%
  gap                                  : +0 points

  -> flat at ceiling. This context (396 tokens) is too small to stress the
     model. Reported honestly: no effect observed at this size.


### 4. Making the effect visible another way

If the retrieval task was too easy, we can raise the difficulty without raising
the token count: ask the model to use **two** facts, one near the front and one
near the back, plus a distractor that looks almost like the needle. Attention has
to span the whole context rather than find one string.

In [7]:
# A near-duplicate distractor: same team, similar name. Now the model cannot
# simply pattern-match on "Logistics" or on an unusual name.
CONFUSER = ("Employee record 99: Wendeline Ott works on the Logistics team and "
            "their internal access code is QRV-247.")

def build_hard(position):
    docs = DISTRACTORS[:position] + [NEEDLE] + DISTRACTORS[position:] + [CONFUSER]
    return "\n".join("Employee record %02d:%s" % (i + 1, d.split(":", 1)[1])
                     for i, d in enumerate(docs))


hard = []
for pos in POSITIONS:
    out = chat([("system", SYSTEM),
                ("user", "Employee records:\n%s\n\n%s" % (build_hard(pos), QUESTION))],
               temperature=0.0, max_tokens=24)
    a = out.content.strip()
    ok = NEEDLE_CODE.lower() in a.lower()
    confused = "247" in a
    hard.append((pos, a, ok, confused))
    print("pos %2d -> %-12s %s%s" % (pos, a[:12], "OK" if ok else "WRONG",
                                     "  (grabbed the look-alike)" if confused else ""))

print("\nhard-variant accuracy: %.0f%%" % (100 * sum(h[2] for h in hard) / len(hard)))

pos  0 -> QRV-742      OK


pos  4 -> QRV-742      OK


pos  8 -> QRV-742      OK


pos 12 -> QRV-742      OK


pos 16 -> QRV-742      OK

hard-variant accuracy: 100%


### 5. What to actually do about it

Three consequences, all cheap to implement:

**1. Rerank, and put the best document first.** This is the highest-value
takeaway. If your retriever returns 15 chunks, the ordering of those 15 is a
free accuracy lever. See `03_rag_advanced/11_reranking`.

**2. Sandwich the instruction.** Put the task instruction at the *start* and
repeat the key constraint at the *end*, after the documents - both are
high-attention positions.

**3. Retrieve less.** Ten mediocre documents are worse than three good ones, not
because of tokens but because the good one is now buried. "Retrieve more, let the
model sort it out" is exactly the strategy this effect punishes.

Let us test consequence 2, since it costs one line of code.

In [8]:
WORST = POSITIONS[len(POSITIONS) // 2]      # the middle slot
ctx = build_context(WORST)

plain = chat([("system", SYSTEM),
              ("user", "Employee records:\n%s\n\n%s" % (ctx, QUESTION))],
             temperature=0.0, max_tokens=24).content.strip()

# Sandwich: question stated BEFORE the records and repeated AFTER them.
sandwiched = chat([("system", SYSTEM),
                   ("user", "%s\n\nEmployee records:\n%s\n\nAgain: %s"
                            % (QUESTION, ctx, QUESTION))],
                  temperature=0.0, max_tokens=24).content.strip()

print("needle at middle position %d" % WORST)
print("  plain      : %-14s %s" % (plain, NEEDLE_CODE.lower() in plain.lower()))
print("  sandwiched : %-14s %s" % (sandwiched, NEEDLE_CODE.lower() in sandwiched.lower()))
print()
print("Extra cost of sandwiching: %d tokens." % approx_tokens(QUESTION))

needle at middle position 8
  plain      : QRV-742        True
  sandwiched : QRV-742        True

Extra cost of sandwiching: 20 tokens.


### 6. Pitfalls

- **Distractors that are too different.** If your filler is lorem ipsum, the
  needle stands out and you measure nothing. Distractors must be confusable.
- **One trial per position.** With a stochastic model, n=1 is noise. Even our
  n=3 is thin - it is a demonstration, not a paper.
- **Testing at a small context and concluding the effect is fake.** The effect
  scales with how full the window is. Test at the size you actually run at.
- **Fixing it with a bigger model.** Bigger windows move the curve, they do not
  flatten it. Ordering is free; a bigger model is not.

### Recap

| Idea | Takeaway |
|---|---|
| Position matters | Being *in* the window is not the same as being *read* |
| U-shape | Start and end are strong, the middle is weak |
| Rerank | The cheapest fix: move the good document to position 1 |
| Sandwich | Repeat the instruction after the documents |
| Retrieve less | Burying the good chunk costs more than missing chunks |

**Next:** [05_structured_vs_prose](05_structured_vs_prose.ipynb) - the same facts,
written two ways, measured for both token cost and accuracy.